# LoRA / QLoRA end-to-end project: teaching a small LLM to write SQL

**Goal:** Fine-tune `Qwen/Qwen2.5-1.5B-Instruct` so that, given a database table schema and an English question, it replies with the correct SQL query.

**Method:** QLoRA = the base model is loaded in **4-bit** (frozen) + small trainable **LoRA adapters** (matrices A and B) are added on top.

**What you will do in this notebook**

| Step | What happens |
|---|---|
| 1 | Set up Colab: GPU, libraries, Google Drive, Hugging Face login |
| 2 | One config cell with every setting you might want to change |
| 3 | Load the base model in 4-bit |
| 4 | Load and format the dataset |
| 5 | Measure the base model **before** training (baseline) |
| 6 | Configure LoRA |
| 7 | Train (checkpoints saved to Drive) |
| 8 | Measure the model **after** training and compare |
| 9 | Save the adapter, upload to Hugging Face, merge |
| 10 | Load your saved adapter in a fresh session |
| 11 | Gradio demo app |
| 12 | Experiments to deepen your understanding |

> **Important about Colab:** your notebook is saved, but every file on the Colab machine is **deleted** when the session ends. That is why this notebook saves checkpoints and the final adapter to **Google Drive**.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`. Then run the cells from top to bottom.

## Step 1: Colab setup

### 1.1 Check the GPU
`nvidia-smi` shows which GPU Colab gave you and how much memory it has. On a free T4 you should see about **15 GB**. If you see an error, the runtime is not set to GPU.

In [ ]:
# "!" runs a shell command instead of Python.
# nvidia-smi = NVIDIA's tool that prints the GPU name, memory used / total, and running processes.
!nvidia-smi

### 1.2 Install libraries

| Library | Role in this project |
|---|---|
| `transformers` | Loads the model and tokenizer from Hugging Face |
| `peft` | **P**arameter-**E**fficient **F**ine-**T**uning: adds the LoRA A and B matrices |
| `trl` | Provides `SFTTrainer`, a ready-made training loop for chat-style fine-tuning |
| `bitsandbytes` | Does the 4-bit NF4 quantization (the "Q" in QLoRA) and the paged 8-bit optimizer |
| `accelerate` | Places model parts on the GPU automatically (`device_map="auto"`) |
| `datasets` | Downloads and processes the training data |
| `gradio` | Builds the small web demo at the end |

After installing, if Colab shows a **"Restart session"** button, click it and continue from the next cell.

In [ ]:
# -q = quiet (less log spam), -U = upgrade to the latest version if an older one is preinstalled.
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets gradio

In [ ]:
# Print the versions you ended up with.
# These libraries change their APIs often. If a later cell errors with an "unexpected keyword argument",
# these versions help you search for the fix (or tell someone else which versions you used).
import transformers, peft, trl, bitsandbytes, datasets, torch

print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("trl          :", trl.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("datasets     :", datasets.__version__)
print("GPU available:", torch.cuda.is_available())   # must be True, otherwise switch the runtime to GPU

### 1.3 Connect Google Drive
Everything important (checkpoints, final adapter, results) is written to a folder in your Drive, so it survives disconnects. Colab will pop up a permission window; allow it.

In [ ]:
from google.colab import drive
import os

# Mount = make your Google Drive appear as a normal folder at /content/drive.
drive.mount('/content/drive')

# The project folder inside your Drive. Everything we want to keep goes here.
PROJECT_DIR = "/content/drive/MyDrive/lora_sql_project"

# exist_ok=True: don't crash if the folder already exists (e.g. when you re-run after a disconnect).
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Saving project files to:", PROJECT_DIR)

### 1.4 Log in to Hugging Face (optional, needed for uploading in Step 9)
Create a token at **huggingface.co → Settings → Access Tokens** with **Write** permission and paste it when asked.
The model used here is not gated, so you can skip this cell if you don't plan to upload.

In [ ]:
from huggingface_hub import login

# Opens a small input box. Paste your token there (it is not stored in the notebook).
login()

## Step 2: Configuration (all the knobs in one place)

Every setting you might want to change lives in this one cell. For the experiments in Step 12, you change values **here** and re-run the notebook from Step 3.

In [ ]:
# ----------------------------- Model -----------------------------
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # base model from the Hugging Face Hub
USE_4BIT = True                            # True = QLoRA (4-bit base). False = plain LoRA (fp16 base, uses more memory)

# ----------------------------- Data ------------------------------
DATASET_ID = "b-mc2/sql-create-context"    # columns: question, context (CREATE TABLE ...), answer (SQL)
TRAIN_SIZE = 3000                          # how many examples to train on (bigger = better but slower)
TEST_SIZE  = 200                           # held-out examples the model never trains on
EVAL_N     = 100                           # how many test examples to score before/after training
SEED       = 42                            # fixed random seed so your runs are reproducible

# ----------------------------- LoRA ------------------------------
LORA_R       = 16      # rank r: the "width" of matrices A (r x d) and B (d x r). Bigger = more capacity
LORA_ALPHA   = 32      # scaling: the adapter output is multiplied by alpha / r (here 32/16 = 2)
LORA_DROPOUT = 0.05    # randomly drops 5% of the adapter's input during training to reduce overfitting
TARGET_MODULES = [     # which linear layers inside each transformer block get an A/B pair
    "q_proj", "k_proj", "v_proj", "o_proj",   # attention projections
    "gate_proj", "up_proj", "down_proj",      # MLP (feed-forward) layers
]

# ----------------------------- Training --------------------------
EPOCHS      = 1        # 1 full pass over the training data
BATCH_SIZE  = 4        # examples processed at once on the GPU (lower this if you get out-of-memory)
GRAD_ACCUM  = 4        # add up gradients over 4 batches before one update -> effective batch = 4 x 4 = 16
LR          = 2e-4     # learning rate; LoRA usually likes higher LRs than full fine-tuning
MAX_LEN     = 512      # max tokens per example; longer ones are cut. Lower it if you run out of memory

# A short name for this run, used for folder names so different experiments don't overwrite each other.
RUN_NAME = f"r{LORA_R}_a{LORA_ALPHA}_n{TRAIN_SIZE}_{'4bit' if USE_4BIT else 'fp16'}"
RUN_DIR  = f"{PROJECT_DIR}/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
print("This run will be saved in:", RUN_DIR)

# The instruction we give the model in every prompt, both during training and when testing.
# Using the SAME system prompt everywhere matters: the model learns to respond to exactly this setup.
SYSTEM_PROMPT = "You are a SQL expert. Given a table schema and a question, reply with only the SQL query."

## Step 3: Load the base model in 4-bit (the "Q" in QLoRA)

**What `BitsAndBytesConfig` does, setting by setting**

- `load_in_4bit=True` stores each weight in 4 bits instead of 16, about 4x less memory.
- `bnb_4bit_quant_type="nf4"` uses **NormalFloat4**, whose 16 levels are placed to match the bell-curve shape of neural network weights.
- `bnb_4bit_use_double_quant=True` also compresses the per-block scale factors (saves a bit more memory).
- `bnb_4bit_compute_dtype=torch.float16` means: weights are *stored* in 4-bit, but *unpacked* to fp16 for the actual math. The T4 GPU doesn't support bf16, so we use fp16.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- Tokenizer: converts text <-> token IDs (numbers the model understands) ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Training pads short examples in a batch to the same length, so a padding token must exist.
# Some models don't define one; reusing the end-of-sequence token is the standard fix.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---- Quantization settings (only used when USE_4BIT is True) ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def load_base_model():
    # Wrapped in a function because we load the base model again later (Step 10, merging).
    if USE_4BIT:
        return AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,   # QLoRA: squeeze the frozen base into 4-bit
            dtype=torch.float16,              # the layers that stay un-quantized (embeddings, norms) use fp16.
                                              # Without this, new transformers versions use the model's default
                                              # bf16, which breaks fp16 training on a T4 (older versions: torch_dtype=)
            device_map="auto",                # let accelerate put the layers on the GPU
        )
    else:
        return AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            dtype=torch.float16,              # plain LoRA: base model in 16-bit (older versions: torch_dtype=)
            device_map="auto",
        )

model = load_base_model()

# get_memory_footprint() = bytes used by the model's weights. Compare USE_4BIT=True vs False to see the saving.
print(f"Model weights use {model.get_memory_footprint() / 1e9:.2f} GB of GPU memory")

In [ ]:
# Print the model's structure. Look for names like q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj:
# those are the linear layers we listed in TARGET_MODULES, i.e. where LoRA will attach its A and B matrices.
print(model)

## Step 4: Load and format the dataset

The dataset has three columns:
- `context`: the table schema, as `CREATE TABLE ...` statements
- `question`: the question in English
- `answer`: the correct SQL query

Chat models are trained on **conversations**, so we turn each row into a mini conversation:
`system` (the instruction) → `user` (schema + question) → `assistant` (the SQL answer).
The trainer then learns to produce the assistant part.

In [ ]:
from datasets import load_dataset

# Download the full training split from the Hugging Face Hub (cached after the first run).
raw = load_dataset(DATASET_ID, split="train")
print(raw)                    # shows the number of rows and the column names
print("\nOne raw example:")
print(raw[0])

In [ ]:
# shuffle(seed=...) mixes the rows randomly but reproducibly; select(range(n)) keeps the first n rows.
# We only take TRAIN_SIZE + TEST_SIZE rows so training stays short on a free GPU.
subset = raw.shuffle(seed=SEED).select(range(TRAIN_SIZE + TEST_SIZE))

# Split into a training part and a test part the model never sees during training.
# Scoring on unseen data is the only honest way to know if the model really learned the skill.
split = subset.train_test_split(test_size=TEST_SIZE, seed=SEED)
train_raw, test_raw = split["train"], split["test"]
print(f"Train: {len(train_raw)} rows | Test: {len(test_raw)} rows")

In [ ]:
def build_user_message(context, question):
    # One place that defines how the prompt looks, reused for training, testing and the demo.
    # Keeping the format identical everywhere is important: the model learns this exact layout.
    return f"Schema:\n{context}\n\nQuestion: {question}"

def to_messages(example):
    # Converts one dataset row into the chat format that SFTTrainer understands.
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": build_user_message(example["context"], example["question"])},
        {"role": "assistant", "content": example["answer"]},   # this is what the model learns to generate
    ]}

# .map() applies to_messages to every row. remove_columns drops the old columns so only "messages" remains.
train_ds = train_raw.map(to_messages, remove_columns=train_raw.column_names)
test_ds  = test_raw.map(to_messages,  remove_columns=test_raw.column_names)

# ALWAYS look at one fully formatted example before training.
# apply_chat_template adds the model's special tokens (e.g. <|im_start|>) exactly as the model saw them in pre-training.
print(tokenizer.apply_chat_template(train_ds[0]["messages"], tokenize=False))

In [ ]:
# How long are the examples in tokens? If many are longer than MAX_LEN they get cut off,
# which can remove the answer. This check tells you if MAX_LEN is big enough.
lengths = [len(tokenizer.apply_chat_template(ex["messages"], tokenize=True)) for ex in train_ds.select(range(500))]
print(f"Token length (500 samples): max={max(lengths)}, average={sum(lengths)/len(lengths):.0f}")
print(f"Examples longer than MAX_LEN={MAX_LEN}: {sum(l > MAX_LEN for l in lengths)}")

## Step 5: Baseline, how good is the model *before* training?

If you don't measure first, you can't prove LoRA helped. We use two scores:

- **Exact match:** the generated SQL equals the correct SQL (after lower-casing and removing extra spaces and the trailing `;`). Strict: `COUNT(*)` vs `count(name)` counts as wrong.
- **Valid SQL rate:** the query runs without error on an empty SQLite database built from the schema. This checks the model writes real, runnable SQL with correct table and column names.

In [ ]:
import re, sqlite3

def ask(model, context, question, max_new_tokens=128):
    # 1) Build the conversation WITHOUT the answer; the model must write it.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_message(context, question)},
    ]
    # 2) Turn it into text with special tokens. add_generation_prompt=True appends the
    #    "assistant is speaking now" marker so the model knows it's its turn.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # 3) Text -> token IDs, moved to the same device (GPU) as the model.
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # 4) Generate. do_sample=False = greedy decoding (always pick the most likely token),
    #    so results are deterministic and fair to compare. no_grad = no gradients needed, saves memory.
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    # 5) The output contains prompt + answer; slice off the prompt tokens and decode only the new ones.
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def clean_sql(text):
    # Base models often wrap the answer in ```sql ... ``` fences or add chatter.
    # We extract just the SQL so both models are judged on the query itself (same cleaning for both = fair).
    fence = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S | re.I)
    if fence:
        text = fence.group(1)
    return text.strip()

def normalize(sql):
    # Lower-case, collapse all whitespace to single spaces, drop a trailing semicolon.
    # So "SELECT  name FROM t;" and "select name from t" count as the same.
    return re.sub(r"\s+", " ", sql.strip().rstrip(";")).lower().strip()

def is_valid_sql(context, sql):
    # Create an empty in-memory SQLite database, run the CREATE TABLE statements from the schema,
    # then try to run the generated query. If SQLite raises an error, the SQL is broken
    # (bad syntax, wrong table/column name, etc.). Tables are empty, so we only check that it RUNS.
    try:
        conn = sqlite3.connect(":memory:")
        conn.executescript(context)
        conn.execute(sql)
        conn.close()
        return True
    except Exception:
        return False

def evaluate(model, dataset, n=EVAL_N, show=3):
    # Runs the model on n test examples and returns (exact_match_rate, valid_sql_rate, predictions).
    model.eval()                       # switch off dropout etc. for testing
    model.config.use_cache = True      # speeds up generation (it's turned off during training)
    exact = valid = 0
    preds = []
    for i, ex in enumerate(dataset.select(range(n))):
        pred = clean_sql(ask(model, ex["context"], ex["question"]))
        preds.append(pred)
        exact += normalize(pred) == normalize(ex["answer"])
        valid += is_valid_sql(ex["context"], pred)
        if i < show:   # print a few examples so you can eyeball the output, not just the numbers
            print(f"Q: {ex['question']}\n  predicted: {pred}\n  correct  : {ex['answer']}\n")
    return exact / n, valid / n, preds

In [ ]:
# Try one hand-written example first so you can see how the untrained model behaves.
print(ask(model,
          "CREATE TABLE employees (name VARCHAR, age INTEGER, dept VARCHAR)",
          "How many employees in the Sales department are older than 30?"))

In [ ]:
# Score the base model on the held-out test set. This takes a few minutes (one generation per example).
base_exact, base_valid, base_preds = evaluate(model, test_raw)
print(f"BASELINE -> exact match: {base_exact:.1%} | valid SQL: {base_valid:.1%}")

## Step 6: Configure LoRA

For every layer named in `TARGET_MODULES`, PEFT keeps the original weight **W frozen** and adds two small trainable matrices:

- **A** (r × d): squeezes the input down to r numbers
- **B** (d × r): expands it back, starting at **all zeros**, so at step 0 the model behaves exactly like the original

The layer's output becomes: `h = W·x + (alpha / r) · B·A·x`

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=LORA_R,                       # rank of A and B
    lora_alpha=LORA_ALPHA,          # scaling numerator (alpha / r)
    lora_dropout=LORA_DROPOUT,      # regularization on the adapter path
    target_modules=TARGET_MODULES,  # which layers get adapters
    bias="none",                    # don't train bias terms; only A and B are trainable
    task_type="CAUSAL_LM",          # tells PEFT this is a next-token-prediction (text generation) model
)
print(lora_config)

## Step 7: Train

`SFTTrainer` (Supervised Fine-Tuning Trainer) handles the whole loop for you: tokenizing the conversations, batching, the forward pass, computing the loss, backpropagating into A and B only, and the optimizer step.

Because we pass `peft_config`, the trainer wraps the model with LoRA itself, and for a 4-bit model it also prepares it for k-bit training (for example, enabling input gradients needed by gradient checkpointing).

**Expected time on a free T4:** roughly 20–40 minutes for 3,000 examples.

**Reading the logs:** `loss` = training loss, `eval_loss` = loss on the test set.
- Both going down: the model is learning.
- `loss` going down but `eval_loss` going **up**: overfitting (memorizing instead of learning).

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=f"{RUN_DIR}/checkpoints",     # checkpoints go to Drive, so a disconnect doesn't lose progress
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,  # simulate a bigger batch without needing more GPU memory
    learning_rate=LR,
    lr_scheduler_type="cosine",              # LR starts high and smoothly decays toward 0 by the end
    warmup_steps=10,                         # first 10 steps: LR ramps up from 0 (avoids a shaky start)
    logging_steps=10,                        # print the training loss every 10 steps
    eval_strategy="steps",                   # compute eval_loss on the test set ...
    eval_steps=50,                           # ... every 50 steps
    save_strategy="steps",
    save_steps=50,                           # save a checkpoint every 50 steps
    save_total_limit=2,                      # keep only the 2 newest checkpoints (saves Drive space)
    fp16=True,                               # mixed precision: faster and less memory on T4
    optim="paged_adamw_8bit",                # QLoRA's paged optimizer: 8-bit state that can spill to CPU RAM on spikes
    gradient_checkpointing=True,             # recompute activations during backward instead of storing them -> less memory
    gradient_checkpointing_kwargs={"use_reentrant": False},   # the recommended, more compatible checkpointing mode
    max_length=MAX_LEN,                      # cut examples longer than this many tokens (older TRL: max_seq_length)
    report_to="none",                        # don't send logs to external tools like Weights & Biases
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,                 # the 4-bit (or fp16) base model from Step 3
    args=args,
    train_dataset=train_ds,      # chat-formatted training data from Step 4
    eval_dataset=test_ds,        # held-out data for eval_loss
    peft_config=lora_config,     # the trainer adds the LoRA A/B matrices using this config
    processing_class=tokenizer,  # (older TRL versions call this argument "tokenizer")
)

# fp16=True uses a "gradient scaler", which only works when the TRAINABLE weights (LoRA A and B) are float32.
# Depending on library versions, the adapters can be created in fp16 or bf16, which crashes training with
# errors like "not implemented for 'BFloat16'" or "Attempting to unscale FP16 gradients".
# Casting just the trainable weights to float32 fixes it. They are tiny, so the extra memory is negligible,
# and the frozen 4-bit base model is untouched. (The optimizer is created at trainer.train(), so it sees fp32 weights.)
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

# Sanity check: every trainable weight should now be float32.
print("Trainable dtypes:", {p.dtype for p in trainer.model.parameters() if p.requires_grad})

# Shows how many numbers are actually being trained vs the model's total. Expect well under 2%.
trainer.model.print_trainable_parameters()

In [ ]:
# Start training. The progress bar shows steps; the table shows loss and eval_loss.
trainer.train()

# ---------------------------------------------------------------------------------
# COLAB DISCONNECTED MID-TRAINING?
# 1) Re-run Step 1 (setup + Drive), Step 2 (config), Step 3 (model), Step 4 (data), Step 6 (LoRA)
#    and the SFTConfig/SFTTrainer cell above.
# 2) Then, instead of trainer.train(), run:
#        trainer.train(resume_from_checkpoint=True)
#    It finds the newest checkpoint in RUN_DIR/checkpoints on Drive and continues from there.
# ---------------------------------------------------------------------------------

In [ ]:
# Plot the loss curves recorded during training, so you can see learning (or overfitting) at a glance.
import matplotlib.pyplot as plt

history = trainer.state.log_history                            # list of dicts logged during training
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts  = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*train_pts), label="training loss")
if eval_pts:
    plt.plot(*zip(*eval_pts), marker="o", label="eval loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title(f"Loss curves ({RUN_NAME})")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(f"{RUN_DIR}/loss_curve.png", dpi=120, bbox_inches="tight")   # also saved to Drive
plt.show()

## Step 8: Evaluate *after* training and compare

We score the fine-tuned model on the exact same test examples as the baseline.

A nice LoRA trick: `with ft_model.disable_adapter():` temporarily switches the adapter off, turning the model back into the original base model without reloading anything. We use it to show the same question answered with and without LoRA.

In [ ]:
ft_model = trainer.model   # the base model + trained LoRA adapters

ft_exact, ft_valid, ft_preds = evaluate(ft_model, test_raw)

print("=" * 55)
print(f"{'':12} | {'exact match':>12} | {'valid SQL':>10}")
print(f"{'base model':12} | {base_exact:>12.1%} | {base_valid:>10.1%}")
print(f"{'fine-tuned':12} | {ft_exact:>12.1%} | {ft_valid:>10.1%}")
print("=" * 55)

In [ ]:
# Same question, adapter OFF vs ON. disable_adapter() is a context manager: inside the "with" block
# LoRA is bypassed (pure base model); after the block it's automatically switched back on.
ctx = "CREATE TABLE students (name VARCHAR, grade INTEGER, city VARCHAR)"
q   = "List the names of students from Hyderabad with a grade above 80."

with ft_model.disable_adapter():
    print("WITHOUT LoRA:", ask(ft_model, ctx, q))
print("WITH LoRA   :", ask(ft_model, ctx, q))

In [ ]:
# Save the scores and settings of this run to Drive, so you can compare experiments later (Step 12).
import json

results = {
    "run_name": RUN_NAME, "model": MODEL_ID, "use_4bit": USE_4BIT,
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "target_modules": TARGET_MODULES,
    "train_size": TRAIN_SIZE, "epochs": EPOCHS, "lr": LR,
    "base_exact": base_exact, "base_valid": base_valid,
    "ft_exact": ft_exact, "ft_valid": ft_valid,
    "train_runtime_sec": trainer.state.log_history[-1].get("train_runtime"),
    "peak_gpu_mem_gb": torch.cuda.max_memory_allocated() / 1e9,   # highest GPU memory used in this session
}
with open(f"{RUN_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## Step 9: Save, upload and merge

### 9.1 Save the adapter to Drive
Only the small A and B matrices are saved, usually a few tens of MB, not the multi-GB base model.

In [ ]:
ADAPTER_DIR = f"{RUN_DIR}/final_adapter"

# save_pretrained on a PEFT model writes ONLY the adapter weights + adapter_config.json.
ft_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)   # save the tokenizer too, so the folder is self-contained

# Show the saved files and their sizes. Notice how tiny the adapter is compared to the base model.
for f in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6
    print(f"{f:40} {size_mb:8.2f} MB")

### 9.2 Upload to Hugging Face (optional, great for your portfolio)
Replace `your-username` with your Hugging Face username. Requires the login from Step 1.4 with a **Write** token.

In [ ]:
HF_REPO = "your-username/qwen2.5-1.5b-text-to-sql-lora"   # <- change this

# Creates the repo (if needed) and uploads the adapter + tokenizer.
ft_model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Uploaded: https://huggingface.co/{HF_REPO}")

### 9.3 Merge the adapter into a standalone model (optional)
Merging computes `W_new = W + (alpha / r) · B·A` once and saves a normal model with no adapter, so inference has zero extra cost.

You must merge into a **16-bit** copy of the base model, not the 4-bit one, so we reload it in fp16. The merged model is about 3 GB.
Free up memory first; if you get out-of-memory here, restart the session and run only Steps 1, 2 and this cell.

In [ ]:
import gc
from peft import PeftModel

# Free GPU memory held by the training objects before loading another copy of the model.
del trainer, ft_model, model
gc.collect(); torch.cuda.empty_cache()

# 1) Load the base model in fp16 (NOT 4-bit) because merging needs full-precision weights to add into.
base16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
# 2) Attach the saved adapter from Drive.
merged = PeftModel.from_pretrained(base16, ADAPTER_DIR)
# 3) merge_and_unload(): folds B·A into W and removes the LoRA wrappers -> a plain transformers model.
merged = merged.merge_and_unload()

MERGED_DIR = f"{RUN_DIR}/merged_model"
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to:", MERGED_DIR)

# Quick sanity check: the merged model should answer like the fine-tuned one.
print(ask(merged, "CREATE TABLE cars (brand VARCHAR, price INTEGER)", "What is the average price of Toyota cars?"))

del merged, base16
gc.collect(); torch.cuda.empty_cache()

## Step 10: Use your trained adapter in a fresh session

Tomorrow, after Colab has wiped the machine, you don't need to retrain. Run **Step 1** (setup + Drive), **Step 2** (config, so paths and settings match), the helper cell from **Step 5** (`ask`, `clean_sql`, ...), and then this cell.

In [ ]:
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the frozen base model (4-bit or fp16 depending on USE_4BIT) ...
base = load_base_model() if "load_base_model" in globals() else AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, dtype=torch.float16, device_map="auto")
# ... and plug the saved adapter from Drive on top. (You can also pass HF_REPO to load from the Hub.)
model = PeftModel.from_pretrained(base, f"{RUN_DIR}/final_adapter")
model.eval()

print(ask(model, "CREATE TABLE orders (id INTEGER, amount INTEGER, status VARCHAR)",
          "What is the total amount of orders with status 'shipped'?"))

## Step 11: Gradio demo app

Gradio builds a small web page around a Python function. `share=True` gives you a public link (valid for about 72 hours) that you can open on your phone or send to friends. It uses `model` from Step 10.

In [ ]:
import gradio as gr

def generate_sql(schema, question):
    # The function Gradio calls whenever the user clicks Submit.
    if not schema.strip() or not question.strip():
        return "Please enter both a schema and a question."
    return clean_sql(ask(model, schema, question))

demo = gr.Interface(
    fn=generate_sql,
    inputs=[
        gr.Textbox(lines=4, label="Table schema (CREATE TABLE ...)",
                   value="CREATE TABLE employees (name VARCHAR, age INTEGER, dept VARCHAR, salary INTEGER)"),
        gr.Textbox(lines=2, label="Question", value="What is the highest salary in the IT department?"),
    ],
    outputs=gr.Code(language="sql", label="Generated SQL"),
    title="Text-to-SQL with LoRA",
    description="Qwen2.5-1.5B fine-tuned with QLoRA on sql-create-context.",
)
demo.launch(share=True, debug=False)

## Step 12: Experiments (where the real understanding comes from)

Change **one** setting in the Step 2 config cell, then re-run from Step 3. Each run saves its own `results.json` in its own folder (the folder name comes from `RUN_NAME`), so nothing gets overwritten.

| Experiment | Change in Step 2 | What you'll learn |
|---|---|---|
| Rank | `LORA_R = 4`, then `64` (keep `LORA_ALPHA = 2 * LORA_R`) | How much capacity the adapter needs; when bigger stops helping |
| Which layers | `TARGET_MODULES = ["q_proj", "v_proj"]` | Whether attention-only adapters are enough, or MLP layers matter |
| Scaling | `LORA_ALPHA = LORA_R`, then `4 * LORA_R` | How alpha / r acts like a volume knob on the adapter |
| Learning rate | `LR = 5e-5`, then `1e-3` | How sensitive LoRA is to the learning rate |
| Data size | `TRAIN_SIZE = 500`, then `10000` | How much data this task really needs |
| QLoRA vs LoRA | `USE_4BIT = False` | The trade-off between memory, speed and quality |

Run the next cell any time to see all your experiments side by side.

In [ ]:
# Collect results.json from every run folder in your Drive project and show them as one table.
import glob
import pandas as pd

rows = []
for path in glob.glob(f"{PROJECT_DIR}/*/results.json"):
    with open(path) as f:
        rows.append(json.load(f))

if rows:
    df = pd.DataFrame(rows)[["run_name", "lora_r", "lora_alpha", "train_size", "use_4bit",
                             "base_exact", "ft_exact", "ft_valid", "train_runtime_sec", "peak_gpu_mem_gb"]]
    display(df.sort_values("ft_exact", ascending=False).round(3))
else:
    print("No results yet. Finish Step 8 at least once.")

## Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA out of memory` | Lower `BATCH_SIZE` to 2 or 1 and raise `GRAD_ACCUM` to keep the effective batch at 16. Or lower `MAX_LEN` to 384. Then `Runtime → Restart session`. |
| `unexpected keyword argument 'max_length'` | Older TRL: rename it to `max_seq_length`. |
| `unexpected keyword argument 'processing_class'` | Older TRL: rename it to `tokenizer`. |
| `unexpected keyword argument 'eval_strategy'` | Older transformers: rename it to `evaluation_strategy`. |
| `not implemented for 'BFloat16'` or `Attempting to unscale FP16 gradients` | The trainable LoRA weights aren't float32. Make sure the model loads with `dtype=torch.float16` (Step 3) and the float32 cast in the trainer cell ran. Restart the session and re-run from Step 1. |
| `unexpected keyword argument 'dtype'` | Older transformers: rename it to `torch_dtype`. |
| `GPU available: False` | `Runtime → Change runtime type → T4 GPU`. |
| No GPU available right now | Free Colab GPUs aren't guaranteed; try again later. |
| Session disconnected during training | See the comment in the training cell: `trainer.train(resume_from_checkpoint=True)`. |
| Gradio link doesn't open | Re-run the demo cell; share links expire after about 72 hours. |

**Next ideas:** try a 3B model (QLoRA still fits on a T4), fine-tune on your own dataset, or write a model card on your Hugging Face page describing your before/after results.